# Práctica 1 — ReAct + tools desde cero

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase04/notebooks/react_tools_scratch.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Objetivo:** construir el loop **ReAct** a mano, en Python puro, para
> **desmitificarlo** — ver que el LLM solo *razona y propone acciones*, y que
> somos nosotros quienes ejecutamos las herramientas y le devolvemos las
> observaciones. Después rehacemos exactamente la misma idea con **LangGraph**
> para ver qué agrega un framework por encima del loop manual.
>
> **Estructura:**
> - **Parte A** — El loop ReAct a mano (parseo de texto `Thought/Action/Observation`).
> - **Parte B** — Lo mismo con LangGraph (`StateGraph` + `ToolNode` + `tools_condition`).
>
> Corre con **Groq** (gratis). Búsqueda web real opcional con **Tavily**;
> sin esa key usamos un *mock* y el notebook corre igual.

## 0. Setup

Instalá dependencias y configurá la API key de Groq.

In [ ]:
!pip install -q groq tavily-python python-dotenv langgraph langchain langchain-groq

In [ ]:
# OPCIONAL — sólo para correr en local. En Colab esta celda se autodetecta y se omite
# (en Colab configurá GROQ_API_KEY desde "Secrets" en la barra lateral izquierda).
#
# Si vos corrés esta notebook localmente con tu venv:
#   1. Poné GROQ_API_KEY=tu-api-key en un archivo .env en la raíz del repo
#   2. Esta celda lo carga automáticamente

try:
    import google.colab  # noqa: F401
    print('ℹ️  Colab detectado — saltando .env (usá Colab Secrets para GROQ_API_KEY).')
except ImportError:
    try:
        from dotenv import load_dotenv, find_dotenv
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
        from dotenv import load_dotenv, find_dotenv
    env_path = find_dotenv(usecwd=True)
    if env_path:
        load_dotenv(env_path)
        print(f'✓ .env cargado desde {env_path}')
    else:
        print('ℹ️  No se encontró .env. Asegurate de exportar GROQ_API_KEY como env var antes de lanzar jupyter.')

In [ ]:
import os

try:
    from google.colab import userdata
    if 'GROQ_API_KEY' not in os.environ:
        try:
            os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
        except Exception:
            pass
    # Tavily es opcional: la intentamos cargar pero no fallamos si no está.
    if 'TAVILY_API_KEY' not in os.environ:
        try:
            os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')
        except Exception:
            pass
except ImportError:
    pass

assert os.environ.get('GROQ_API_KEY'), (
    'Falta GROQ_API_KEY. Configurala en Colab Secrets o como env var.'
)
print('✓ GROQ_API_KEY configurada')

**Sobre `TAVILY_API_KEY` (opcional):** si la configurás (Colab Secrets o `.env`),
la herramienta `buscar_web` hace búsqueda web **real** con [Tavily](https://tavily.com).
Si no está, usamos un **mock** con respuestas predefinidas y el notebook corre igual —
ideal para clase, porque la traza es determinística.

# Parte A — El loop ReAct a mano

La idea central de ReAct (Yao et al., 2022) es **intercalar razonamiento y acción**:

```
Thought:  el modelo razona qué hacer
Action:   propone una herramienta + argumento
Observation: nosotros ejecutamos la tool y le devolvemos el resultado
... (se repite) ...
Answer:   el modelo da la respuesta final
```

**El punto clave que vamos a ver con nuestras propias manos:**

> El LLM **NO ejecuta** las herramientas. Solo **genera texto** que dice
> `Action: calculadora[2+2]`. Nuestro código lo **parsea**, ejecuta la función
> Python de verdad, y le **devuelve** el resultado como `Observation:`. El LLM es
> el cerebro; nosotros somos las manos.

Vamos a construir esto en ~30 líneas.

## A.1 — Las herramientas (funciones Python comunes)

Dos tools, nada mágico: son funciones que reciben un string y devuelven un string.

In [ ]:
import ast
import operator

# ── Tool 1: calculadora ─────────────────────────────────────────────
# Eval "seguro": en vez de usar eval() (que ejecutaría CUALQUIER código Python),
# parseamos la expresión con ast y solo permitimos nodos aritméticos.
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    if isinstance(node, ast.Constant):          # números (3.8+)
        if not isinstance(node.value, (int, float)):
            raise ValueError('solo se permiten números')
        return node.value
    if isinstance(node, ast.BinOp):             # a + b, a * b, ...
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp):           # -a, +a
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f'expresión no permitida: {ast.dump(node)}')


def calculadora(expr: str) -> str:
    """Evalúa una expresión aritmética simple de forma segura.

    Solo admite números y los operadores + - * / ** % y signo unario.
    Nunca ejecuta código arbitrario (no usa eval()).
    """
    try:
        arbol = ast.parse(expr.strip(), mode='eval')
        return str(_eval_node(arbol.body))
    except Exception as e:
        return f'ERROR: no pude evaluar "{expr}" ({e})'


# Probamos la tool sola, sin LLM:
print(calculadora('2026 - 1573'))
print(calculadora('(30 * 9/5) + 32'))
print(calculadora('import os'))   # debe rechazarse

In [ ]:
# ── Tool 2: buscar_web ──────────────────────────────────────────────
# Si hay TAVILY_API_KEY, búsqueda real; si no, un mock determinístico.

_MOCK_BUSQUEDA = {
    'santa fe fundacion': (
        'La ciudad de Santa Fe (Santa Fe de la Vera Cruz) fue fundada por '
        'Juan de Garay en el año 1573.'
    ),
    'default': (
        '[MOCK] No hay TAVILY_API_KEY configurada, así que esto es una respuesta '
        'simulada. En un entorno real, acá vendría el resultado de la búsqueda web.'
    ),
}


def _buscar_mock(query: str) -> str:
    q = query.lower()
    for clave, texto in _MOCK_BUSQUEDA.items():
        if clave == 'default':
            continue
        # match laxo: si todas las palabras de la clave aparecen en la query
        if all(p in q for p in clave.split()):
            return f'[MOCK] {texto}'
    return _MOCK_BUSQUEDA['default']


def buscar_web(query: str) -> str:
    """Busca en la web y devuelve un resumen breve.

    Usa Tavily si TAVILY_API_KEY está configurada; si no, un mock.
    """
    api_key = os.environ.get('TAVILY_API_KEY')
    if not api_key:
        return _buscar_mock(query)
    try:
        from tavily import TavilyClient
        client = TavilyClient(api_key=api_key)
        resp = client.search(query=query, max_results=3, include_answer=True)
        if resp.get('answer'):
            return resp['answer']
        # fallback: concatenamos los snippets de los resultados
        trozos = [r.get('content', '') for r in resp.get('results', [])]
        return ' '.join(trozos)[:600] or '(sin resultados)'
    except Exception as e:
        # Si Tavily falla por cualquier motivo, caemos al mock para no romper la demo.
        return f'{_buscar_mock(query)} (Tavily falló: {e})'


# El registro de tools: nombre -> función. El loop ReAct lo usa para despachar.
TOOLS = {
    'calculadora': calculadora,
    'buscar_web': buscar_web,
}

print(buscar_web('¿En qué año se fundó Santa Fe?'))

## A.2 — El wrapper del LLM (Groq)

Una sola función que manda mensajes a Groq y devuelve el texto. Nada más.

In [ ]:
from groq import Groq

_groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])


def llamar_llm(messages, model='llama-3.3-70b-versatile', temperature=0.0, max_tokens=700) -> str:
    """Wrapper mínimo sobre Groq chat completions.

    model: usamos 'llama-3.3-70b-versatile' (bueno siguiendo formatos).
           Alternativas en Groq: 'llama-3.1-8b-instant' (más rápido/barato),
           'openai/gpt-oss-20b', 'qwen/qwen3-32b'.
    Para correr local podrías cambiar este wrapper por Ollama
    (p.ej. ollama.chat(model='qwen3:8b', messages=messages)).
    Usamos temperature=0.0 para que la traza sea lo más determinística posible.
    """
    resp = _groq_client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return resp.choices[0].message.content


print(llamar_llm([{'role': 'user', 'content': 'Decime "ok" si me podés escuchar'}]))

## A.3 — El system prompt que enseña el formato ReAct

Acá está el corazón del truco: le **enseñamos al modelo el formato exacto** que
queremos que produzca, y le listamos las herramientas disponibles. El modelo va a
imitar ese formato porque es buen seguidor de instrucciones.

In [ ]:
SYSTEM_REACT = """Sos un asistente que resuelve preguntas razonando paso a paso y usando herramientas.

Tenés que responder SIEMPRE en este formato, una línea por vez:

Thought: <tu razonamiento sobre qué hacer a continuación>
Action: <nombre_herramienta>[<argumento>]

Después de cada Action, el sistema va a ejecutar la herramienta y te va a dar:
Observation: <resultado de la herramienta>

Podés repetir el ciclo Thought/Action/Observation las veces que necesites.
Cuando ya tengas todo para responder, terminá con:

Thought: <razonamiento final>
Answer: <la respuesta final para el usuario>

Herramientas disponibles:
- calculadora[expresion]  -> evalúa una expresión aritmética. Ej: calculadora[2026 - 1573]
- buscar_web[consulta]    -> busca información en la web. Ej: buscar_web[año fundación de Santa Fe]

Reglas:
- Una sola Action por turno. NO escribas la Observation vos mismo: la pone el sistema.
- Para cualquier cálculo numérico usá SIEMPRE la calculadora, no calcules de cabeza.
- Para datos que no sabés con certeza (fechas, hechos), usá buscar_web.
- No inventes resultados de herramientas."""

print(SYSTEM_REACT)

## A.4 — El parser y el loop

`parse_action` extrae con una regex el `Action: tool[arg]`. `react_agent` arma el
loop: llama al LLM, mira si terminó (`Answer:`) o pidió una acción (`Action:`),
ejecuta la tool, agrega la `Observation:` a la conversación y repite.

Imprimimos cada paso para que **veas la traza** completa.

In [ ]:
import re

# Captura:  Action: nombre[argumento]   (el argumento puede tener corchetes adentro,
# así que tomamos hasta el ÚLTIMO ] de la línea con un match no codicioso al nombre).
_RE_ACTION = re.compile(r'Action:\s*([a-zA-Z_]\w*)\s*\[(.*)\]', re.DOTALL)
_RE_ANSWER = re.compile(r'Answer:\s*(.*)', re.DOTALL)


def parse_action(text):
    """Devuelve (nombre_tool, argumento) si hay un Action, o None."""
    m = _RE_ACTION.search(text)
    if not m:
        return None
    return m.group(1).strip(), m.group(2).strip()


def parse_answer(text):
    """Devuelve la respuesta final si hay un Answer, o None."""
    m = _RE_ANSWER.search(text)
    return m.group(1).strip() if m else None


def react_agent(pregunta, max_steps=6):
    """Loop ReAct a mano. Devuelve la respuesta final (o None si se agotaron los pasos)."""
    messages = [
        {'role': 'system', 'content': SYSTEM_REACT},
        {'role': 'user', 'content': pregunta},
    ]
    print(f'❓ Pregunta: {pregunta}\n' + '─' * 70)

    for paso in range(1, max_steps + 1):
        # 1) El LLM razona y propone (texto, NO ejecuta nada).
        #    Cortamos en "Observation:" por si el modelo intenta alucinarla.
        salida = llamar_llm(messages + [{'role': 'system',
                                         'content': 'Generá SOLO hasta la próxima Action o el Answer. No escribas Observation.'}])
        if 'Observation:' in salida:
            salida = salida.split('Observation:')[0].strip()
        print(f'[Paso {paso}]\n{salida}')

        # 2) ¿Terminó?
        respuesta = parse_answer(salida)
        if respuesta is not None and parse_action(salida) is None:
            print('─' * 70)
            print(f'✅ Answer: {respuesta}')
            return respuesta

        # 3) ¿Pidió una acción? La ejecutamos NOSOTROS.
        accion = parse_action(salida)
        if accion is None:
            # El modelo no respetó el formato; le pedimos que siga.
            messages.append({'role': 'assistant', 'content': salida})
            messages.append({'role': 'user', 'content':
                             'No detecté ni Action ni Answer. Seguí el formato: Thought/Action o Answer.'})
            continue

        nombre, arg = accion
        tool = TOOLS.get(nombre)
        if tool is None:
            observacion = f'ERROR: no existe la herramienta "{nombre}". Usá: {list(TOOLS)}'
        else:
            observacion = tool(arg)   # <-- ACÁ se ejecuta la función Python de verdad

        print(f'   🔧 ejecuto {nombre}[{arg}]  ->  Observation: {observacion}\n')

        # 4) Realimentamos la observación a la conversación y seguimos el loop.
        messages.append({'role': 'assistant', 'content': salida})
        messages.append({'role': 'user', 'content': f'Observation: {observacion}'})

    print('⚠️  Se agotaron los pasos sin llegar a un Answer.')
    return None

## A.5 — Demo: una pregunta que necesita las DOS herramientas

`buscar_web` para el año de fundación, y `calculadora` para la resta. Mirá cómo el
modelo encadena los pasos solo, y cómo cada `Observation` la ponemos nosotros.

In [ ]:
_ = react_agent(
    '¿En qué año se fundó la ciudad de Santa Fe y cuántos años pasaron hasta 2026?'
)

## A.6 — Reflexión

Eso es todo: **un agente ReAct cabe en ~30 líneas**. El LLM razona y *propone*
acciones como texto; nuestro código las parsea, ejecuta las herramientas y le
devuelve las observaciones. **El modelo es el cerebro; nosotros somos las manos.**

Lo frágil acá es el **parseo de texto**: si el modelo no respeta el formato, todo
se rompe. Por eso los modelos modernos traen *tool calling* nativo (devuelven JSON
estructurado en vez de texto a parsear) y los frameworks lo encapsulan. Eso es lo
que vemos en la Parte B.

# Parte B — ReAct con LangGraph

Misma idea, pero ahora con un framework. ¿Qué nos da **LangGraph**?

- **Estado** explícito (la lista de mensajes) con un *reducer* que acumula.
- **Tool calling nativo**: en vez de parsear texto, el modelo devuelve llamadas a
  herramientas estructuradas (`tool_calls`), y un `ToolNode` las ejecuta.
- **Control de flujo declarativo**: un grafo con nodos (`agent`, `tools`) y aristas
  condicionales (`tools_condition`) en vez de un `while` a mano.

No es magia: por debajo es el **mismo loop ReAct**. Solo que el framework se encarga
del parseo, el estado y el ruteo.

## B.1 — Las tools, ahora con el decorador `@tool`

Reusamos exactamente la misma lógica de la Parte A, pero envuelta con `@tool` para
que LangChain/LangGraph conozca su nombre, su descripción (¡del docstring!) y el
schema de argumentos. Esa metadata es lo que el modelo ve para decidir cuál usar.

In [ ]:
from langchain_core.tools import tool


@tool
def calculadora_tool(expr: str) -> str:
    """Evalúa una expresión aritmética simple (ej: '2026 - 1573'). Usala para cualquier cálculo."""
    return calculadora(expr)   # reusamos la función segura de la Parte A


@tool
def buscar_web_tool(query: str) -> str:
    """Busca información en la web (fechas, hechos, datos que no sabés con certeza)."""
    return buscar_web(query)   # reusamos la función de la Parte A


tools_lg = [calculadora_tool, buscar_web_tool]
print('Tools registradas:', [t.name for t in tools_lg])

## B.2 — El modelo con tools enlazadas

`ChatGroq` es el wrapper de LangChain para Groq. Con `.bind_tools(...)` le decimos
qué herramientas tiene disponibles; a partir de ahí el modelo puede devolver
`tool_calls` estructurados.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0)
# Alternativas Groq: 'llama-3.1-8b-instant', 'openai/gpt-oss-20b', 'qwen/qwen3-32b'.
llm_con_tools = llm.bind_tools(tools_lg)

print('✓ LLM con tools enlazadas')

## B.3 — El grafo: `agent` ⇄ `tools`

Construimos un `StateGraph` sobre `MessagesState` (estado = lista de mensajes con un
reducer que acumula). Dos nodos:

- **`agent`**: llama al LLM. Si el modelo quiere usar una tool, su respuesta trae `tool_calls`.
- **`tools`**: un `ToolNode` prearmado que ejecuta esas tool_calls y agrega las observaciones.

`tools_condition` es la arista condicional: si el último mensaje del `agent` tiene
`tool_calls`, va a `tools`; si no, termina. Después de `tools`, volvemos siempre al
`agent`. **Ese ida y vuelta ES el loop ReAct.**

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition


def nodo_agent(state: MessagesState):
    """Nodo del cerebro: llama al LLM con todo el historial de mensajes."""
    return {'messages': [llm_con_tools.invoke(state['messages'])]}


builder = StateGraph(MessagesState)
builder.add_node('agent', nodo_agent)
builder.add_node('tools', ToolNode(tools_lg))   # ejecuta las tool_calls

builder.add_edge(START, 'agent')
# tools_condition rutea: si hay tool_calls -> 'tools', si no -> END
builder.add_conditional_edges('agent', tools_condition)
builder.add_edge('tools', 'agent')              # vuelta al cerebro: el loop ReAct

grafo = builder.compile()
print('✓ Grafo compilado')

# (Opcional) ver el diagrama del grafo:
try:
    from IPython.display import Image, display
    display(Image(grafo.get_graph().draw_mermaid_png()))
except Exception:
    print(grafo.get_graph().draw_ascii())

## B.4 — Lo corremos sobre la misma pregunta

Le pasamos la pregunta como un mensaje de usuario dentro del estado. El grafo itera
solo (agent → tools → agent → ...) hasta que el modelo responde sin pedir más tools.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

estado_final = grafo.invoke({'messages': [
    SystemMessage(content='Para cualquier cálculo numérico usá SIEMPRE la calculadora. '
                          'Para fechas o hechos que no sepas con certeza, usá la búsqueda web.'),
    HumanMessage(content='¿En qué año se fundó la ciudad de Santa Fe y cuántos años pasaron hasta 2026?'),
]})

# Imprimimos la traza completa de mensajes (es el equivalente a nuestra traza manual)
for m in estado_final['messages']:
    m.pretty_print()

## B.5 — Comparación final: loop manual vs LangGraph

| | **Parte A (a mano)** | **Parte B (LangGraph)** |
|---|---|---|
| Cómo pide tools el LLM | texto `Action: tool[arg]` que parseamos con regex | `tool_calls` estructurados (JSON nativo) |
| Quién ejecuta la tool | nuestro `if`/dispatch sobre `TOOLS` | el `ToolNode` prearmado |
| Estado | una lista `messages` que mutamos a mano | `MessagesState` con reducer |
| Control de flujo | un `while` con `max_steps` | grafo + `tools_condition` |
| Robustez | frágil (depende del formato de texto) | robusta (tool calling nativo) |

> **El mensaje:** por debajo es **el mismo loop ReAct** — razonar, actuar, observar,
> repetir. El framework agrega **estado, estructura y un tool-node**, no inteligencia
> nueva. Entender la Parte A es entender lo que LangGraph hace por vos en la Parte B.
>
> En la **Práctica 2** subimos un nivel: varios agentes coordinados (patrón
> orquestador) con LangGraph, reusando estas mismas herramientas.